# Orochi Model Testing with Real Data

**Data Location:** `/group/jug/aman/orochi/data/` (`.parquet` format)

**Pretrained Checkpoints:**
- `pretrained_checkpoints/mamba_fm_3d.pth.tar`
- `pretrained_checkpoints/MambaULight_epoch_99_loss_-0.0624.pth.tar`

This notebook demonstrates:
1. Loading data from `.parquet` files
2. Initializing the Orochi 3D model
3. Loading pretrained weights
4. Running inference on real biomedical images
5. Visualizing and evaluating results

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add package to path
package_root = Path.cwd()
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from types import SimpleNamespace
import warnings
warnings.filterwarnings('ignore')

# Import Orochi
from orochi.models import MambaEncoderHeria, reg_decoder
from orochi.losses import get_loss_function
from orochi.metrics import get_metric

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ Device: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Data Configuration

In [ ]:
# Data paths
DATA_ROOT = Path('/group/jug/aman/orochi/data')
CHECKPOINT_DIR = Path('pretrained_checkpoints')

# Available datasets (adjust based on your actual data structure)
DATASETS = {
    'hipsc_2d': DATA_ROOT / 'hipsc_2d',
    'hipsc_3d': DATA_ROOT / 'hipsc_3d',
    'hipct_2d': DATA_ROOT / 'hipct_2d',
    'idr_2d': DATA_ROOT / 'idr_2d',
}

print("Data Root:", DATA_ROOT)
print("\nChecking available datasets:")
for name, path in DATASETS.items():
    exists = path.exists() if isinstance(path, Path) else False
    status = "✓" if exists else "✗"
    print(f"  {status} {name}: {path}")

# List parquet files
if DATA_ROOT.exists():
    parquet_files = list(DATA_ROOT.glob('**/*.parquet'))
    print(f"\nFound {len(parquet_files)} .parquet files")
    if len(parquet_files) > 0:
        print("First 5:")
        for f in parquet_files[:5]:
            print(f"  - {f.relative_to(DATA_ROOT)}")

## Load Data from Parquet

In [ ]:
def load_parquet_data(parquet_path, num_samples=1):
    """
    Load image data from parquet file.
    
    Adjust this function based on your actual parquet structure.
    """
    # Read parquet file
    df = pd.read_parquet(parquet_path)
    
    print(f"Parquet file loaded: {parquet_path.name}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Shape: {df.shape}")
    print(f"  First row keys: {df.iloc[0].keys().tolist() if len(df) > 0 else 'N/A'}")
    
    return df

# Try to load first available parquet file
if DATA_ROOT.exists():
    parquet_files = list(DATA_ROOT.glob('**/*.parquet'))
    
    if len(parquet_files) > 0:
        # Load first file as example
        sample_file = parquet_files[0]
        print(f"Loading sample: {sample_file.relative_to(DATA_ROOT)}\n")
        
        try:
            df = load_parquet_data(sample_file)
            print("\n✓ Successfully loaded parquet file")
            print("\nInspect the dataframe to understand the structure:")
            print(df.head(2))
        except Exception as e:
            print(f"⚠ Error loading parquet: {e}")
            print("Please adjust load_parquet_data() function for your data structure")
    else:
        print("⚠ No parquet files found")
else:
    print(f"⚠ Data root not found: {DATA_ROOT}")
    print("Using synthetic data for demonstration...")

## Model Configuration

In [ ]:
# Model config for 3D registration
config = SimpleNamespace(
    dimensions=3,
    img_size=(160, 192, 224),      # Standard brain MRI size (adjust if needed)
    in_chans=2,                     # 2 for registration (moving + fixed)
    embed_dim=96,
    depths=[2, 2, 2, 2],
    num_heads=[3, 6, 12, 24],
    window_size=(5, 6, 7),
    patch_size=4,
    drop_rate=0.0,
    drop_path_rate=0.1,
    use_checkpoint=False,
)

print("Model Configuration:")
for key, value in vars(config).items():
    print(f"  {key}: {value}")

## Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Create model
encoder = MambaEncoderHeria(config).to(device)
decoder = reg_decoder(config).to(device)

# Count parameters
enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
total = enc_params + dec_params

print(f"✓ Model initialized:")
print(f"  Encoder: {enc_params:,} parameters")
print(f"  Decoder: {dec_params:,} parameters")
print(f"  Total: {total:,} parameters ({total*4/1024**2:.1f} MB)")

## Load Pretrained Weights

In [ ]:
# Available checkpoints
CHECKPOINTS = [
    'mamba_fm_3d.pth.tar',
    'MambaULight_epoch_99_loss_-0.0624.pth.tar'
]

# Select checkpoint (change index to use different one)
checkpoint_name = CHECKPOINTS[0]
checkpoint_path = CHECKPOINT_DIR / checkpoint_name

print(f"Loading: {checkpoint_path}\n")

if checkpoint_path.exists():
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Inspect checkpoint structure
    print(f"Checkpoint keys: {list(checkpoint.keys())}")
    
    # Extract state dict (handle different checkpoint formats)
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    elif 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
        state_dict = checkpoint['model']
    else:
        state_dict = checkpoint
    
    # Show some state dict keys
    print(f"\nState dict keys (first 5):")
    for i, key in enumerate(list(state_dict.keys())[:5]):
        print(f"  {key}: {state_dict[key].shape}")
    
    # Load weights
    try:
        missing, unexpected = encoder.load_state_dict(state_dict, strict=False)
        print(f"\n✓ Weights loaded successfully!")
        if len(missing) > 0:
            print(f"  Missing keys: {len(missing)}")
        if len(unexpected) > 0:
            print(f"  Unexpected keys: {len(unexpected)}")
    except Exception as e:
        print(f"\n⚠ Error loading weights: {e}")
        print("Continuing with random initialization...")
    
    # Show additional checkpoint info
    if 'epoch' in checkpoint:
        print(f"  Epoch: {checkpoint['epoch']}")
    if 'loss' in checkpoint:
        print(f"  Loss: {checkpoint['loss']}")
        
else:
    print(f"⚠ Checkpoint not found: {checkpoint_path}")
    available = list(CHECKPOINT_DIR.glob('*.pth*')) if CHECKPOINT_DIR.exists() else []
    print(f"Available: {[f.name for f in available]}")
    print("\nContinuing with random initialization...")

## Prepare Test Data

Convert your parquet data to tensors or use synthetic data

In [ ]:
# Option 1: Load from parquet (adjust based on your data structure)
# Uncomment and modify once you understand your parquet structure
#
# def parquet_to_tensor(df, idx=0):
#     """
#     Convert parquet row to tensor.
#     Adjust this based on your actual data format.
#     """
#     # Example (modify for your data):
#     # row = df.iloc[idx]
#     # image_data = row['image']  # or however images are stored
#     # tensor = torch.from_numpy(image_data).float()
#     # return tensor
#     pass

# Option 2: Use synthetic test data
print("Creating synthetic test data...\n")

batch_size = 1
D, H, W = config.img_size

# Generate test volumes
moving = torch.randn(batch_size, 1, D, H, W, device=device)
fixed = torch.randn(batch_size, 1, D, H, W, device=device)

# Normalize to [0, 1]
moving = (moving - moving.min()) / (moving.max() - moving.min())
fixed = (fixed - fixed.min()) / (fixed.max() - fixed.min())

print(f"✓ Test data prepared:")
print(f"  Moving: {moving.shape}")
print(f"  Fixed: {fixed.shape}")
print(f"  Range: [{moving.min():.3f}, {moving.max():.3f}]")

# Option 3: Load real medical images (NIfTI)
# import nibabel as nib
# moving_nii = nib.load('path/to/moving.nii.gz')
# moving = torch.from_numpy(moving_nii.get_fdata()).unsqueeze(0).unsqueeze(0).float().to(device)

## Run Inference

In [ ]:
encoder.eval()
decoder.eval()

print("Running inference...\n")

with torch.no_grad():
    # Concatenate inputs
    combined = torch.cat([moving, fixed], dim=1)
    print(f"Input: {combined.shape}")
    
    # Forward pass
    features = encoder(combined)
    print(f"Features: {len(features)} levels")
    for i, f in enumerate(features):
        print(f"  Level {i}: {f.shape}")
    
    # Decode
    flow = decoder(features)
    print(f"\nOutput flow: {flow.shape}")
    print(f"Flow range: [{flow.min():.3f}, {flow.max():.3f}]")

print("\n✓ Inference complete!")

## Compute Metrics

In [ ]:
# Simple metrics without warping
mse_fn = get_metric('mse')
mae_fn = get_metric('mae')

with torch.no_grad():
    mse = mse_fn(moving.cpu(), fixed.cpu())
    mae = mae_fn(moving.cpu(), fixed.cpu())

print("Metrics (before registration):")
print(f"  MSE: {mse:.6f}")
print(f"  MAE: {mae:.6f}")
print(f"\nFlow statistics:")
print(f"  Mean: {flow.mean():.6f}")
print(f"  Std: {flow.std():.6f}")
print(f"  Max displacement: {flow.abs().max():.3f}")

## Visualize Results

In [ ]:
# Visualize middle slices
slice_idx = D // 2

moving_slice = moving[0, 0, slice_idx].cpu().numpy()
fixed_slice = fixed[0, 0, slice_idx].cpu().numpy()
flow_z = flow[0, 0, slice_idx].cpu().numpy()
flow_y = flow[0, 1, slice_idx].cpu().numpy()
flow_x = flow[0, 2, slice_idx].cpu().numpy()
flow_mag = np.sqrt(flow_z**2 + flow_y**2 + flow_x**2)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Images
axes[0, 0].imshow(moving_slice, cmap='gray')
axes[0, 0].set_title(f'Moving (slice {slice_idx})')
axes[0, 0].axis('off')

axes[0, 1].imshow(fixed_slice, cmap='gray')
axes[0, 1].set_title(f'Fixed (slice {slice_idx})')
axes[0, 1].axis('off')

diff = np.abs(moving_slice - fixed_slice)
im = axes[0, 2].imshow(diff, cmap='hot')
axes[0, 2].set_title('Difference')
axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046)

# Flow components
im = axes[1, 0].imshow(flow_z, cmap='RdBu_r')
axes[1, 0].set_title('Flow Z')
axes[1, 0].axis('off')
plt.colorbar(im, ax=axes[1, 0], fraction=0.046)

im = axes[1, 1].imshow(flow_y, cmap='RdBu_r')
axes[1, 1].set_title('Flow Y')
axes[1, 1].axis('off')
plt.colorbar(im, ax=axes[1, 1], fraction=0.046)

im = axes[1, 2].imshow(flow_mag, cmap='viridis')
axes[1, 2].set_title('Flow Magnitude')
axes[1, 2].axis('off')
plt.colorbar(im, ax=axes[1, 2], fraction=0.046)

plt.tight_layout()
plt.savefig('results_visualization.png', dpi=150, bbox_inches='tight')
print("✓ Saved: results_visualization.png")
plt.show()

## Save Results

In [ ]:
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

# Save results
torch.save({
    'moving': moving.cpu(),
    'fixed': fixed.cpu(),
    'flow': flow.cpu(),
    'config': vars(config),
    'checkpoint': checkpoint_name,
}, output_dir / 'inference_results.pth')

print(f"✓ Results saved to: {output_dir}/inference_results.pth")

## Summary

In [ ]:
print("="*70)
print("INFERENCE SUMMARY")
print("="*70)
print(f"\nModel: MambaEncoderHeria (3D)")
print(f"Parameters: {total:,}")
print(f"Checkpoint: {checkpoint_name}")
print(f"\nInput size: {config.img_size}")
print(f"Output: 3D flow field {flow.shape}")
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU Memory Used: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
print("\n" + "="*70)
print("\nNext steps:")
print("1. Modify load_parquet_data() for your actual data structure")
print("2. Implement parquet_to_tensor() conversion")
print("3. Run on real biomedical images")
print("4. See docs/ folder for more information")